# Lab | Search by Meaning, by Hand

Goal: turn a small knowledge base into embeddings, turn a few questions into
embeddings, and find the best-matching passages by computing **cosine
similarity ourselves with NumPy** — no vector store, no built-in `.search()`.


## Setup

Install dependencies (run once):

```
pip install -r requirements.txt
```

We use `sentence-transformers` locally, so no API key is required.

In [1]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded.")

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 553.07it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.


## Step 1 — Load the knowledge base and embed every passage

We load `knowledge_base.json`, embed every passage's `text` **once**, and
keep each vector together with its `id` and `source` so we can trace a
result back to where it came from.

In [2]:
with open("knowledge_base.json", "r") as f:
    knowledge_base = json.load(f)

print(f"Loaded {len(knowledge_base)} passages.")

texts = [doc["text"] for doc in knowledge_base]
passage_embeddings = model.encode(texts)  # shape: (num_passages, embedding_dim)

print(f"Embedding shape: {passage_embeddings.shape}")

indexed_passages = []
for doc, vector in zip(knowledge_base, passage_embeddings):
    indexed_passages.append({
        "id": doc["id"],
        "source": doc["source"],
        "text": doc["text"],
        "vector": vector,
    })

# Sanity check
indexed_passages[0]["id"], indexed_passages[0]["vector"][:5]

Loaded 10 passages.
Embedding shape: (10, 384)


('kb-01',
 array([ 0.06680017,  0.02304333,  0.01242329, -0.01783404, -0.0724795 ],
       dtype=float32))

## Step 2 — Cosine similarity, written by hand

$$\text{cosine\_similarity}(a, b) = \frac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}$$

The dot product alone grows just because a vector is "longer", not because
it points in a more similar direction. Dividing by the magnitudes (L2
norms) strips that out, so we're purely comparing **direction** — which is
what carries meaning in an embedding space.

In [3]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)


# Quick sanity tests before trusting this on real embeddings
v1 = np.array([1.0, 2.0, 3.0])
v2 = np.array([-1.0, -2.0, -3.0])      # opposite direction
v3 = np.array([2.0, 4.0, 6.0])         # same direction, different length

print("identical vectors:           ", cosine_similarity(v1, v1))   # -> 1.0
print("opposite vectors:            ", cosine_similarity(v1, v2))   # -> -1.0
print("same direction, diff length: ", cosine_similarity(v1, v3))   # -> 1.0

identical vectors:            1.0
opposite vectors:             -1.0
same direction, diff length:  1.0


## Step 3 — Embed each query and rank passages by similarity

For each query we: embed it with the **same** model used for the passages,
score it against every passage with our hand-written `cosine_similarity`,
then sort and keep the top 3.

In [5]:
def search(query: str, top_k: int = 3):
    query_vector = model.encode(query)

    scored = []
    for passage in indexed_passages:
        score = cosine_similarity(query_vector, passage["vector"])
        scored.append((score, passage))

    scored.sort(key=lambda pair: pair[0], reverse=True)
    return scored[:top_k]


test_queries = [
    "my laptop won't switch on",
    "how do I stop being billed every month?",
    "access denied error when saving a file",
    "where do I leave my car in the evening?",
]

for query in test_queries:
    print("=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)
    for rank, (score, passage) in enumerate(search(query), start=1):
        print(f"  #{rank}  score={score:.4f}  [{passage['id']} | {passage['source']}]")
        print(f"        {passage['text']}")
    print()

QUERY: my laptop won't switch on
  #1  score=0.4369  [kb-02 | handbook.md]
        To power up a device that won't turn on, hold the power button for ten seconds, then connect the charger and wait two minutes before trying again.
  #2  score=0.1774  [kb-07 | it.md]
        Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security.
  #3  score=0.1581  [kb-09 | it.md]
        Company laptops back up automatically to the cloud every night at 2am while connected to the office network. Files in the Desktop and Documents folders are included; external drives are not.

QUERY: how do I stop being billed every month?
  #1  score=0.5408  [kb-05 | policy.md]
        To cancel your subscription, open Account Settings and choose End Plan. Cancellation takes effect at the end of the current billing period; no partial refunds are given for the remaining days.
  #2  score=0.3041  [kb-06 | policy.m

## Reflection

For each query, did the best match share any words with the query? What
does that tell you about what the embedding captured?

- **"my laptop won't switch on"** → top match talks about a *device* that
  *won't turn on* and how to *power it up*. "Laptop" and "switch on" never
  appear in the passage, yet the meaning lines up almost exactly.
- **"how do I stop being billed every month?"** → top match is about
  *cancelling a subscription* and the *billing period*. "Stop being billed"
  and "cancel your subscription" are different words for the same action.
- **"access denied error when saving a file"** → top match is the
  *error code 0x80070005 ("access denied")* passage. "Saving a file" isn't
  mentioned, but the embedding still connects the error scenario.
- **"where do I leave my car in the evening?"** → top match is the
  *parking lot* policy. "Car" and "leave" never appear in the passage —
  "park" captures the same idea.

**Takeaway:** none of the top matches needed exact word overlap with the
query. The embedding model captured the *underlying meaning/intent* of
each sentence, not its literal vocabulary — which is exactly what makes
semantic search useful for things like support tickets or natural-language
questions, where users rarely phrase things the same way the docs do.

## Optional stretch — a query with no good answer

Let's add a query that the knowledge base genuinely doesn't cover, and look
at its top score compared to the others above.

In [6]:
stretch_query = "what's the wifi password?"

print(f"QUERY: {stretch_query}")
for rank, (score, passage) in enumerate(search(stretch_query), start=1):
    print(f"  #{rank}  score={score:.4f}  [{passage['id']} | {passage['source']}]")
    print(f"        {passage['text']}")

QUERY: what's the wifi password?
  #1  score=0.3186  [kb-07 | it.md]
        Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security.
  #2  score=0.1227  [kb-09 | it.md]
        Company laptops back up automatically to the cloud every night at 2am while connected to the office network. Files in the Desktop and Documents folders are included; external drives are not.
  #3  score=0.1209  [kb-02 | handbook.md]
        To power up a device that won't turn on, hold the power button for ten seconds, then connect the charger and wait two minutes before trying again.


**Observation:** the top match was `kb-07` (password reset) with a score
of **0.3186** — noticeably lower than the top scores for the four real
queries above (0.4369–0.5437). That makes sense: the knowledge base never
discusses wifi or network credentials. The model still found *some* signal
because "password" appears in `kb-07`, but it's a surface-level word match,
not a real answer to "what's the wifi password" — password *reset* and
wifi password are different things entirely.

**How a threshold could help:** the four genuine queries all scored above
**0.43**, while the unanswerable query topped out at **0.3186** — a clear
gap. If we set a threshold somewhere in between, say **0.40**, the system
could check the top score before answering: above the threshold, return the
passage with confidence; below it, respond with something like "I couldn't
find relevant information for this question" instead of confidently
returning a weak, unrelated passage. With only one negative example this
threshold is just a rough estimate — in practice you'd tune it using many
more known-good and known-bad queries before trusting it in production.